### Tools 
Models can request to call tools that performs tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. Parrots are birds known for their ability to mimic human speech. I think it\'s something related to their vocal apparatus, maybe their syrinx? I remember that the syrinx is the vocal organ in birds, similar to the human larynx. So maybe parrots can produce a variety of sounds because their syrinx is structured in a way that allows for complex sounds. \n\nBut why do they do it? Is it just mimicry, or is there a purpose? I\'ve heard that some parrots, like the African Grey, are particularly good at imitating human speech. Maybe it\'s a way of social bonding? Parrots are social animals, so they might copy sounds to communicate with each other or with humans. In the wild, they might mimic other bird calls or environmental sounds. When kept as pets, they copy human speech. \n\nI also wonder if it\'s a survival mechanism. Perhaps by mimicking sounds, they can bette

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather ar a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("Whats's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    #View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'xmz743fat', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.145651891, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.006254988, 'prompt_tokens_details': None, 'queue_time': 0.054394658, 'total_time': 0.151906879}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f11f9-4ff4-7aa0-a

### Tools Execution Loops

In [7]:
# Step 1: Model generates tool calls
messages = [{"role":"user", "content":"What's the weather in Boston"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute tools and collect results
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
#"The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny. ☀️ Let me know if you need more details!


In [8]:
messages

[{'role': 'user', 'content': "What's the weather in Boston"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Boston is the location they mentioned. So I\'ll call get_weather with location set to "Boston". Make sure the JSON is correctly formatted. No other parameters needed. Just the name and arguments. Alright, that should do it.\n', 'tool_calls': [{'id': '5e2se9sph', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 152, 'total_tokens': 262, 'completion_time': 0.1818215, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 0.007702045, 'prompt_tokens_details': None, 'queue_time': 0.054767495, 'total_time': 0.189523545}, 'model_name': 'qwe